In [1]:
import xarray as xr
import yaml
import os
import sys

def read_config():
    config_path = 'local_config_rwanda.yaml'
    try:
        with open(config_path, 'r') as f:
            try:
                localconfig = yaml.safe_load(f)
            except yaml.YAMLError as e:
                print(e)
                sys.exit(1)
            return localconfig
    except FileNotFoundError as e:
        print(e)
        print("You must set local_config.yaml in the main folder. Copy local_config-example.yaml and adjust appropriately.")
        sys.exit(1)


def get_data_paths():
    data_config_paths = 'data_paths.yaml'

    try:
        with open(data_config_paths, 'r') as f:
            try:
                all_data_paths = yaml.safe_load(f)
            except yaml.YAMLError as e:
                print(e)
                sys.exit(1)
    except FileNotFoundError as e:
        print(e)
        print("data_paths.yaml not found. Should exist in main folder")
        sys.exit(1)

    lc = read_config()
    data_paths = all_data_paths[lc['data_paths']]
    return data_paths

/home/m/matthewwright/nobackups/anaconda3/envs/initial-analysis/lib/python3.10/site-packages/xarray/backends/cfgrib_.py:29: UserWarning: Failed to load cfgrib - most likely there is a problem accessing the ecCodes library. Try `import cfgrib` to get the full error message
  warnings.warn(


In [2]:
fcst_yaml_path = "forecast_example.yaml"

In [3]:
with open(fcst_yaml_path, "r") as f:
    try:
        fcst_params = yaml.safe_load(f)
    except yaml.YAMLError as exc:
        print(exc)
        raise

output_folder = fcst_params["OUTPUT"]["folder"]
save_crps_only = fcst_params["OUTPUT"]["save_crps_only"]
truth_input_folder = fcst_params["INPUT"]["truth_folder"]

In [4]:
files = os.listdir(output_folder)

In [6]:
file_path = files[10] # Pick a random file

fcst = xr.load_dataset(f"{output_folder}/{file_path}")
fcst

<xarray.Dataset>
Dimensions:           (y: 396, x: 396, member: 50, time: 1, valid_time: 1)
Coordinates:
  * y                 (y) float32 -1.98e+05 -1.97e+05 ... 1.96e+05 1.97e+05
  * x                 (x) float32 -1.98e+05 -1.97e+05 ... 1.96e+05 1.97e+05
  * member            (member) int32 1 2 3 4 5 6 7 8 ... 43 44 45 46 47 48 49 50
  * time              (time) datetime64[ns] 2026-06-20
Dimensions without coordinates: valid_time
Data variables:
    fcst_valid_time   (time, valid_time) datetime64[ns] 2026-06-21T06:00:00
    precipitation     (time, member, valid_time, y, x) float32 0.1559 ... 0.3699
    crps              (time, valid_time, y, x) float32 nan nan nan ... nan nan
    radar_centre_lat  float32 -2.158
    radar_centre_lon  float32 30.11
    radar_lat         (y, x) float32 -3.947 -3.947 -3.947 ... -0.3757 -0.3757
    radar_lon         (y, x) float32 28.33 28.34 28.35 ... 31.86 31.87 31.88
Attributes:
    description:  GAN 24-hour rainfall ensemble members in the ICPAC region.

In [ ]:
#plot precip if present
if not save_crps_only:
    fcst.precipitation.median('member').isel(valid_time=0, time=0).plot()

In [ ]:
#plot truth
time = fcst.fcst_valid_time[0]
truth = xr.load_dataset(f"{truth_input_folder}/{str(time.dt.year.values[0])}/{time.dt.strftime('%Y%m%d').values[0]}_06.nc")
truth.rainfall_rate.isel(time=0).plot()


In [ ]:
#plot crps
fcst.crps.isel(valid_time=0, time=0).plot()

In [ ]:
#Plot CRPS
fcsts = xr.open_mfdataset(f"{output_folder}/*.nc", combine="by_coords")
fcsts = fcsts.where(fcsts.crps < 1e36) #remove missing values

In [ ]:
fcsts.crps.mean('time').plot()

In [ ]:
fcsts.crps.mean(('x','y')).plot()